# Membuat Edges (Hubungan Guru dan Murid)

## Import Library

In [1]:
import pandas as pd
import ast
import re

## Load Data

In [2]:
# pd.set_option('display.max_colwidth', None)

df = pd.read_csv('../data/processed/sanadset_cleaned.csv', header=None)
df.columns = ['Hadith', 'Book', 'Num_hadith', 'Matn', 'Sanad', 'Sanad_Length', 'Sanad_no_harakat']
df = df.iloc[1:].reset_index(drop=True)


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_23048\1150346939.py:3: DtypeWarning: Columns (0: 2, 1: 5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/processed/sanadset_cleaned.csv', header=None)


### Menghitung data dan Memecah isi list pada kolom Sanad_no_harakat

In [3]:
df['Sanad_no_harakat'] = df['Sanad_no_harakat'].apply(ast.literal_eval)


In [4]:
df.shape

(487451, 7)

In [5]:
print(type(df['Sanad_no_harakat'].iloc[0]))

<class 'list'>


In [6]:
sanad_exploded = df.explode('Sanad_no_harakat')
display(sanad_exploded.head())


,Hadith,Book,Num_hadith,Matn,Sanad,Sanad_Length,Sanad_no_harakat
0,حَدَّثَنِي <SANAD> <NAR> أَبُو عُبَيْدَةَ مُسْ...,مسند الربيع بن حبيب,1,، عَنِ النَّبِيِّ صَلَّى اللَّهُ عَلَيْهِ وَ...,['أَبُو عُبَيْدَةَ مُسْلِمُ بْنُ أَبِي كَرِيمَ...,3,ابو عبيده مسلم بن ابو كريمه التميمي
0,حَدَّثَنِي <SANAD> <NAR> أَبُو عُبَيْدَةَ مُسْ...,مسند الربيع بن حبيب,1,، عَنِ النَّبِيِّ صَلَّى اللَّهُ عَلَيْهِ وَ...,['أَبُو عُبَيْدَةَ مُسْلِمُ بْنُ أَبِي كَرِيمَ...,3,جابر بن زيد الازدي
0,حَدَّثَنِي <SANAD> <NAR> أَبُو عُبَيْدَةَ مُسْ...,مسند الربيع بن حبيب,1,، عَنِ النَّبِيِّ صَلَّى اللَّهُ عَلَيْهِ وَ...,['أَبُو عُبَيْدَةَ مُسْلِمُ بْنُ أَبِي كَرِيمَ...,3,عبد الله بن عباس
1,حَدَّثَنِي <SANAD> <NAR> أَبُو عُبَيْدَةَ </NA...,مسند الربيع بن حبيب,3,، أُمِّ الْمُؤْمِنِينَ ، رَضِيَ اللَّهُ عَنْ...,"['أَبُو عُبَيْدَةَ', 'جَابِرِ بْنِ زَيْدٍ', 'ع...",3,ابو عبيده
1,حَدَّثَنِي <SANAD> <NAR> أَبُو عُبَيْدَةَ </NA...,مسند الربيع بن حبيب,3,، أُمِّ الْمُؤْمِنِينَ ، رَضِيَ اللَّهُ عَنْ...,"['أَبُو عُبَيْدَةَ', 'جَابِرِ بْنِ زَيْدٍ', 'ع...",3,جابر بن زيد


In [7]:
sanad_exploded.shape

(2993196, 7)

In [8]:
df.Sanad_no_harakat.head(10)

0    [ابو عبيده مسلم بن ابو كريمه التميمي, جابر بن ...
1                      [ابو عبيده, جابر بن زيد, عائشه]
2                             [ابو عبيده, جابر بن زيد]
3                  [ابو عبيده, جابر بن زيد, ابو هريره]
4            [ابو عبيده, جابر بن زيد, ابو سعيد الخدري]
5                   [ابو عبيده, جابر بن زيد, ابن عباس]
6                [ابو عبيده, جابر بن زيد, انس بن مالك]
7            [ابو عبيده, جابر بن زيد, ابو سعيد الخدري]
8                  [ابو عبيده, جابر بن زيد, ابو هريره]
9              [ابو عبيده, جابر بن زيد, عمر بن الخطاب]
Name: Sanad_no_harakat, dtype: object

### Menghitung Perawi (Narrator) Paling Banyak Muncul

In [9]:
from collections import Counter

# Fungsi normalisasi nama Arab
def normalize_arabic_name(text):
    text = str(text)

    # Hilangkan spasi berlebih
    text = re.sub(r'\s+', ' ', text).strip()

    # Hapus akhiran " L" jika ada
    text = re.sub(r'\s+L$', '', text)

    # Normalisasi huruf Arab
    text = text.replace('أ', 'ا')
    text = text.replace('إ', 'ا')
    text = text.replace('آ', 'ا')

    text = text.replace('ى', 'ي')
    text = text.replace('ة', 'ه')

    return text

# Flatten semua nama perawi dari list Sanad_no_harakat
all_narrators = []

for sanad_list in df['Sanad_no_harakat']:
    if isinstance(sanad_list, list):
        all_narrators.extend(sanad_list)

# Normalisasi nama perawi
all_narrators_normalized = [
    normalize_arabic_name(name)
    for name in all_narrators
]

# Hitung frekuensi setelah normalisasi
narrator_counts = Counter(all_narrators_normalized)

# DataFrame hasil
narrator_df = pd.DataFrame(
    narrator_counts.most_common(),
    columns=['Perawi', 'Jumlah']
)

display(narrator_df)

,Perawi,Jumlah
0,ابو هريره,51669
1,ابن عباس,48300
2,عائشه,33145
3,ابن عمر,28892
4,شعبه,26570
...,...,...
177284,جد بن فضيل,1
177285,ابو سلمه الخرساني,1
177286,الاشعث بن طلق,1
177287,حفص بن موسي,1


In [10]:
mapping = []

for sanad_list in df['Sanad_no_harakat']:
    if isinstance(sanad_list, list):
        for name in sanad_list:
            mapping.append({
                'Nama_Asli': name,
                'Nama_Normalisasi': normalize_arabic_name(name)
            })

mapping_df = pd.DataFrame(mapping)

merged = (
    mapping_df
    .groupby('Nama_Normalisasi')['Nama_Asli']
    .unique()
    .reset_index()
)

merged['Jumlah_Varian'] = merged['Nama_Asli'].apply(len)

merged = merged[
    merged['Jumlah_Varian'] > 1
].sort_values(
    'Jumlah_Varian',
    ascending=False
)

display(merged)

,Nama_Normalisasi,Nama_Asli,Jumlah_Varian


### Buat Hubungan Guru-Murid dari Sanad


In [11]:
from IPython.display import display

# Fungsi membentuk hubungan guru-murid
def build_teacher_student_edges(chain):
    if not isinstance(chain, list) or len(chain) < 2:
        return []
    
    # Format: (Guru, Murid)
    return [(chain[i + 1], chain[i]) for i in range(len(chain) - 1)]

# Pastikan semua data berupa list
def ensure_list(cell):
    if isinstance(cell, list):
        return cell
    
    if isinstance(cell, str) and cell.strip().startswith('['):
        try:
            return ast.literal_eval(cell)
        except Exception:
            return [cell]
    
    if pd.isna(cell):
        return []
    
    return [cell]

# Ubah kolom menjadi list
sanad_lists = df['Sanad_no_harakat'].apply(ensure_list)

# Membentuk semua hubungan guru-murid
edges = []
for sanad in sanad_lists:
    edges.extend(build_teacher_student_edges(sanad))

# Membuat DataFrame hubungan
edges_df = pd.DataFrame(edges, columns=['Guru', 'Murid'])

# Hitung weight tanpa mengubah urutan awal
edges_df['Weight'] = (
    edges_df
    .groupby(['Guru', 'Murid'])['Guru']
    .transform('count')
)

# Hapus duplikat tetapi tetap menjaga urutan awal
edges_df = edges_df.drop_duplicates(subset=['Guru', 'Murid'])

# Reset index
edges_df = edges_df.reset_index(drop=True)

# Output
display(edges_df[['Guru', 'Murid', 'Weight']])

,Guru,Murid,Weight
0,جابر بن زيد الازدي,ابو عبيده مسلم بن ابو كريمه التميمي,1
1,عبد الله بن عباس,جابر بن زيد الازدي,1
2,جابر بن زيد,ابو عبيده,522
3,عائشه,جابر بن زيد,68
4,ابو هريره,جابر بن زيد,76
...,...,...,...
781056,ابي مسعود البدري,حذيفه بن اليمان,1
781057,سعيد بن العاص,ابي مسعود البدري,1
781058,ابو مسعود,سعيد بن العاص,1
781059,عبد الله,علي بن علقمه,1


In [12]:
type(df['Sanad_no_harakat'].iloc[0])

list

In [13]:
type(edges_df['Guru'].iloc[0])

str

### Menghapus Lafal Periwayatan

In [14]:
# Normalisasi terakhir
def clean_final_name(s):
    if pd.isna(s):
        return ""

    s = str(s)

    s = re.sub(r'\s+', ' ', s).strip()
    s = re.sub(r'\s+L$', '', s)


    # Hapus karakter tersembunyi
    s = re.sub(r'[\u200b-\u200f\u202a-\u202e\u2060\ufeff]', '', s)

    # Sisakan huruf Arab dan spasi
    s = re.sub(r'[^\u0600-\u06FF\s]', ' ', s)

    # Rapikan spasi
    s = re.sub(r'\s+', ' ', s).strip()

    return s


# Cleaning nama
edges_df['Guru'] = edges_df['Guru'].apply(clean_final_name)
edges_df['Murid'] = edges_df['Murid'].apply(clean_final_name)

# Hapus relasi tidak valid
edges_df = edges_df[
    (edges_df['Guru'].str.len() > 0) &
    (edges_df['Murid'].str.len() > 0) &
    (edges_df['Guru'] != edges_df['Murid'])
].copy()

# Gabungkan relasi sama
edges_df = (
    edges_df
    .groupby(['Guru', 'Murid'], as_index=False, sort=False)
    .agg(Weight=('Weight', 'sum'))
)

# Inverse weight
edges_df['Inverse_Weight'] = edges_df['Weight'].apply(
    lambda x: 1 / x if x > 0 else 0
)

# Reset index
edges_df = edges_df.reset_index(drop=True)

display(edges_df)

,Guru,Murid,Weight,Inverse_Weight
0,جابر بن زيد الازدي,ابو عبيده مسلم بن ابو كريمه التميمي,1,1.000000
1,عبد الله بن عباس,جابر بن زيد الازدي,1,1.000000
2,جابر بن زيد,ابو عبيده,522,0.001916
3,عائشه,جابر بن زيد,68,0.014706
4,ابو هريره,جابر بن زيد,76,0.013158
...,...,...,...,...
776793,ابي مسعود البدري,حذيفه بن اليمان,1,1.000000
776794,سعيد بن العاص,ابي مسعود البدري,1,1.000000
776795,ابو مسعود,سعيد بن العاص,1,1.000000
776796,عبد الله,علي بن علقمه,1,1.000000


### Mengecek karakter selain huruf arab

In [15]:
# Cek karakter selain huruf Arab
def check_non_arabic(text):

    if not isinstance(text, str):
        return False

    # Cari karakter selain Arab dan spasi
    return bool(
        re.search(r'[^\u0600-\u06FF\s]', text)
    )

# Ambil data yang masih memiliki
# karakter non-Arab pada kolom Guru atau Murid
non_arabic_df = edges_df[
    edges_df['Guru'].apply(check_non_arabic) |
    edges_df['Murid'].apply(check_non_arabic)
]

# Tampilkan hasil
display(non_arabic_df[['Guru', 'Murid']].head(20))

# Jumlah data
print("Jumlah data dengan karakter non-Arab:")
print(len(non_arabic_df))

,Guru,Murid


Jumlah data dengan karakter non-Arab:
0


### Menghapus karakter selain huruf arab

In [16]:
# Fungsi untuk menghapus karakter selain Arab dan spasi
def remove_non_arabic(text):
    if not isinstance(text, str):
        return text

    # Hapus karakter selain Arab dan spasi
    text = re.sub(r'[^\u0600-\u06FF\s]', '', text)

    # Rapikan spasi berlebih
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Terapkan ke kolom Guru dan Murid
edges_df['Guru'] = edges_df['Guru'].apply(remove_non_arabic)
edges_df['Murid'] = edges_df['Murid'].apply(remove_non_arabic)

# Cek ulang apakah masih ada karakter non-Arab
non_arabic_df = edges_df[
    edges_df['Guru'].apply(check_non_arabic) |
    edges_df['Murid'].apply(check_non_arabic)
]

print("Jumlah data dengan karakter non-Arab setelah dibersihkan:")
print(len(non_arabic_df))

display(edges_df[['Guru', 'Murid']])

Jumlah data dengan karakter non-Arab setelah dibersihkan:
0


,Guru,Murid
0,جابر بن زيد الازدي,ابو عبيده مسلم بن ابو كريمه التميمي
1,عبد الله بن عباس,جابر بن زيد الازدي
2,جابر بن زيد,ابو عبيده
3,عائشه,جابر بن زيد
4,ابو هريره,جابر بن زيد
...,...,...
776793,ابي مسعود البدري,حذيفه بن اليمان
776794,سعيد بن العاص,ابي مسعود البدري
776795,ابو مسعود,سعيد بن العاص
776796,عبد الله,علي بن علقمه


In [17]:
edges_df = edges_df.dropna(subset=['Guru', 'Murid'])

### remove text

In [18]:
remove_text = "أن أم الفضل بنت الحارث بعثته إلى معاوية بالشام"

edges_df = edges_df[
    ~edges_df['Guru'].str.contains(remove_text, na=False) &
    ~edges_df['Murid'].str.contains(remove_text, na=False)
]
edges_df = edges_df.reset_index(drop=True)
edges_df.shape

(776798, 4)

### Membersihkan data hubungan Guru dan Murid jika mengandung angka

In [19]:
edges_df = edges_df[
    ~edges_df['Guru'].str.contains(r'\d', na=False) &
    ~edges_df['Murid'].str.contains(r'\d', na=False)
]
edges_df = edges_df.reset_index(drop=True)
edges_df.shape 

(776798, 4)

In [20]:
# Membersihkan data relasi guru-murid supaya tidak ada relasi yang salah karena salah satu namanya kosong.
invalid_relations = edges_df[
    (edges_df['Guru'] == '') |
    (edges_df['Murid'] == '')
]

print("Jumlah relasi tidak valid (Guru atau Murid kosong):", len(invalid_relations))

display(invalid_relations[['Guru', 'Murid', 'Weight']])

edges_df = edges_df[
    (edges_df['Guru'] != '') &
    (edges_df['Murid'] != '')
]

edges_df = edges_df.reset_index(drop=True)
display(edges_df[['Guru', 'Murid', 'Weight']])
edges_df.shape

Jumlah relasi tidak valid (Guru atau Murid kosong): 0


,Guru,Murid,Weight


,Guru,Murid,Weight
0,جابر بن زيد الازدي,ابو عبيده مسلم بن ابو كريمه التميمي,1
1,عبد الله بن عباس,جابر بن زيد الازدي,1
2,جابر بن زيد,ابو عبيده,522
3,عائشه,جابر بن زيد,68
4,ابو هريره,جابر بن زيد,76
...,...,...,...
776793,ابي مسعود البدري,حذيفه بن اليمان,1
776794,سعيد بن العاص,ابي مسعود البدري,1
776795,ابو مسعود,سعيد بن العاص,1
776796,عبد الله,علي بن علقمه,1


(776798, 4)

In [21]:
# Hilangkan spasi berlebih terlebih dahulu
edges_df['Guru'] = edges_df['Guru'].astype(str).str.strip()
edges_df['Murid'] = edges_df['Murid'].astype(str).str.strip()

# Cari relasi yang sama
same_relation = edges_df[
    edges_df['Guru'] == edges_df['Murid']
]

print('Jumlah relasi guru-murid yang sama:', len(same_relation))

display(same_relation[['Guru', 'Murid', 'Weight']])

# Hapus relasi yang sama
edges_df = edges_df[
    edges_df['Guru'] != edges_df['Murid']
]

# Reset index
edges_df = edges_df.reset_index(drop=True)

print('Jumlah data setelah relasi sama dihapus:', len(edges_df))

display(edges_df[['Guru', 'Murid', 'Weight']])

edges_df.shape
# edges_df = edges_df[
#     edges_df['Guru'] != edges_df['Murid']
# ]
# edges_df = edges_df.reset_index(drop=True)
# edges_df.shape

# display(edges_df[['Guru', 'Murid', 'Weight']])

Jumlah relasi guru-murid yang sama: 0


,Guru,Murid,Weight


Jumlah data setelah relasi sama dihapus: 776798


,Guru,Murid,Weight
0,جابر بن زيد الازدي,ابو عبيده مسلم بن ابو كريمه التميمي,1
1,عبد الله بن عباس,جابر بن زيد الازدي,1
2,جابر بن زيد,ابو عبيده,522
3,عائشه,جابر بن زيد,68
4,ابو هريره,جابر بن زيد,76
...,...,...,...
776793,ابي مسعود البدري,حذيفه بن اليمان,1
776794,سعيد بن العاص,ابي مسعود البدري,1
776795,ابو مسعود,سعيد بن العاص,1
776796,عبد الله,علي بن علقمه,1


(776798, 4)

In [22]:
# Daftar lafaz periwayatan
hapus_kata = [
    "ثنا", "حدثنا", "قال حدثنا", "وقال", "عن",
    "أخبرنا", "أنبأنا", "حدثني", "حدثه", "سمعت",
    "يقول", "ذكر", "رواه", "حدث", "حكى"
]



# Regex
pola = r'\b(?:' + '|'.join(map(re.escape, hapus_kata)) + r')\b'

mengandung_lafaz = edges_df[
    edges_df['Guru'].str.contains(pola, na=False) |
    edges_df['Murid'].str.contains(pola, na=False)
]

display(mengandung_lafaz)
# Hapus lafaz dari kolom Guru dan Murid
edges_df['Guru'] = (
    edges_df['Guru']
    .str.replace(pola, '', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

edges_df['Murid'] = (
    edges_df['Murid']
    .str.replace(pola, '', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

# Tampilkan hasil
display(edges_df)

,Guru,Murid,Weight,Inverse_Weight


,Guru,Murid,Weight,Inverse_Weight
0,جابر بن زيد الازدي,ابو عبيده مسلم بن ابو كريمه التميمي,1,1.000000
1,عبد الله بن عباس,جابر بن زيد الازدي,1,1.000000
2,جابر بن زيد,ابو عبيده,522,0.001916
3,عائشه,جابر بن زيد,68,0.014706
4,ابو هريره,جابر بن زيد,76,0.013158
...,...,...,...,...
776793,ابي مسعود البدري,حذيفه بن اليمان,1,1.000000
776794,سعيد بن العاص,ابي مسعود البدري,1,1.000000
776795,ابو مسعود,سعيد بن العاص,1,1.000000
776796,عبد الله,علي بن علقمه,1,1.000000


In [23]:
# Menampilkan data duplikat beserta karakter tersembunyi

# Ambil semua relasi duplikat
duplikat_df = edges_df[
    edges_df.duplicated(
        subset=['Guru', 'Murid'],
        keep=False
    )
]

# Urutkan agar mudah dibaca
duplikat_df = duplikat_df.sort_values(
    by=['Guru', 'Murid']
).reset_index(drop=True)

# Tampilkan dataframe biasa
display(duplikat_df)


# Menampilkan relasi guru-murid yang duplikat

duplikat_df = edges_df[
    edges_df.duplicated(
        subset=['Guru', 'Murid'],
        keep=False
    )
]

for i, row in duplikat_df.iterrows():
    print(repr(row['Guru']), repr(row['Murid']))

# Urutkan agar mudah dilihat
duplikat_df = duplikat_df.sort_values(
    by=['Guru', 'Murid']
).reset_index(drop=True)

display(duplikat_df)

# Hapus duplikat tetapi tetap menjaga urutan awal
edges_df = edges_df.drop_duplicates(subset=['Guru', 'Murid'])

# Reset index
edges_df = edges_df.reset_index(drop=True)

# Output
display(edges_df[['Guru', 'Murid', 'Weight', 'Inverse_Weight']])

,Guru,Murid,Weight,Inverse_Weight


,Guru,Murid,Weight,Inverse_Weight


,Guru,Murid,Weight,Inverse_Weight
0,جابر بن زيد الازدي,ابو عبيده مسلم بن ابو كريمه التميمي,1,1.000000
1,عبد الله بن عباس,جابر بن زيد الازدي,1,1.000000
2,جابر بن زيد,ابو عبيده,522,0.001916
3,عائشه,جابر بن زيد,68,0.014706
4,ابو هريره,جابر بن زيد,76,0.013158
...,...,...,...,...
776793,ابي مسعود البدري,حذيفه بن اليمان,1,1.000000
776794,سعيد بن العاص,ابي مسعود البدري,1,1.000000
776795,ابو مسعود,سعيد بن العاص,1,1.000000
776796,عبد الله,علي بن علقمه,1,1.000000


In [24]:
def clean_text(s):
    if pd.isna(s):
        return s
    
    # hapus karakter non-printable TAPI jangan hapus huruf Arab
    s = re.sub(r"[^\x20-\x7E\u0600-\u06FF\s]", "", str(s))
    # rapikan spasi
    return re.sub(r"\s+", " ", s).strip()

edges_df['Guru'] = edges_df['Guru'].apply(clean_text)
edges_df['Murid'] = edges_df['Murid'].apply(clean_text)

# Reset index
edges_df = edges_df.reset_index(drop=True)

# Output
display(edges_df[['Guru', 'Murid', 'Weight', 'Inverse_Weight']])

,Guru,Murid,Weight,Inverse_Weight
0,جابر بن زيد الازدي,ابو عبيده مسلم بن ابو كريمه التميمي,1,1.000000
1,عبد الله بن عباس,جابر بن زيد الازدي,1,1.000000
2,جابر بن زيد,ابو عبيده,522,0.001916
3,عائشه,جابر بن زيد,68,0.014706
4,ابو هريره,جابر بن زيد,76,0.013158
...,...,...,...,...
776793,ابي مسعود البدري,حذيفه بن اليمان,1,1.000000
776794,سعيد بن العاص,ابي مسعود البدري,1,1.000000
776795,ابو مسعود,سعيد بن العاص,1,1.000000
776796,عبد الله,علي بن علقمه,1,1.000000


In [25]:
print(edges_df.dtypes)

Guru                  str
Murid                 str
Weight              int64
Inverse_Weight    float64
dtype: object


In [27]:
edges_df.to_csv("../data/processed/edges.csv", index=False, encoding='utf-8-sig')